In [1]:
import pyspark
from google.colab import files


In [2]:
from pyspark.sql import SparkSession

Start spark session


In [3]:
spark = SparkSession.builder\
    .appName("Uber_analysis")\
        .getOrCreate()

Import Files here

In [5]:
df =spark.read.csv('dataset.csv',header =True, inferSchema=True)

In [6]:
df.show()

+---------+------------+---------+-------+----------------+---------+--------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|
+---------+------------+---------+-------+----------------+---------+--------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|
|     NULL|           8|        6|      0|               2|        2|            14|
|     NULL|           9|        8|      3|               0|        0|            14|
|     NULL|          10|        9|      2|               0|        1|            14|
|     NULL|          11|       11|      1|               4|        4|            11|
|     NULL|          12|       12|      0|               2|        2|            11|
|     NULL|          13|        9|      1|               0|        0|             9|
|     NULL|          14|       12|      1|               0|        0|             9|
|     NULL|          15|       11|      2|               1|      

In [7]:
df.filter(df["Date"].isNotNull()).show()


+---------+------------+---------+-------+----------------+---------+--------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|
+---------+------------+---------+-------+----------------+---------+--------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|
|11-Sep-12|           0|        9|      3|               1|        1|             3|
|12-Sep-12|           0|        9|      2|               0|        1|             2|
|13-Sep-12|           0|       11|     11|               0|        2|             0|
|14-Sep-12|           0|       10|      1|               3|        4|             3|
|15-Sep-12|           0|       45|      2|              23|       24|            19|
|16-Sep-12|           0|       44|      2|              17|       20|            15|
|17-Sep-12|           0|       11|      5|               0|        2|             2|
|18-Sep-12|           0|       28|     18|               3|      

#1) Find which date had the most completed trips during the two-week period?

Fill with previous date for the null columns

In [44]:
from pyspark.sql import *
from pyspark.sql.functions import *

In [9]:


# Add row ID to preserve order
df_updated = df.withColumn("row_id", monotonically_increasing_id())

# Create window specification
windowSpec = Window.orderBy("row_id").rowsBetween(Window.unboundedPreceding, 0)

# Forward fill the Date column
df_updated = df_updated.withColumn(
    "Date",
    last(col("Date"), ignorenulls=True).over(windowSpec)
)

# Drop temporary column
df_updated = df_updated.drop("row_id")

df_updated.show()

+---------+------------+---------+-------+----------------+---------+--------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|
+---------+------------+---------+-------+----------------+---------+--------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|
|10-Sep-12|           8|        6|      0|               2|        2|            14|
|10-Sep-12|           9|        8|      3|               0|        0|            14|
|10-Sep-12|          10|        9|      2|               0|        1|            14|
|10-Sep-12|          11|       11|      1|               4|        4|            11|
|10-Sep-12|          12|       12|      0|               2|        2|            11|
|10-Sep-12|          13|        9|      1|               0|        0|             9|
|10-Sep-12|          14|       12|      1|               0|        0|             9|
|10-Sep-12|          15|       11|      2|               1|      

In [10]:
completed_trip = df_updated.groupBy("Date").sum("Completed Trips ")
completed_trip=completed_trip.withColumnRenamed("sum(Completed Trips )","Completed Trips")



In [11]:
completed_trip.show()

+---------+---------------+
|     Date|Completed Trips|
+---------+---------------+
|10-Sep-12|             26|
|11-Sep-12|             40|
|12-Sep-12|             91|
|13-Sep-12|             45|
|14-Sep-12|            108|
|15-Sep-12|            199|
|16-Sep-12|             93|
|17-Sep-12|             57|
|18-Sep-12|             42|
|19-Sep-12|             41|
|20-Sep-12|             70|
|21-Sep-12|            190|
|22-Sep-12|            248|
|23-Sep-12|            111|
|24-Sep-12|              4|
+---------+---------------+



In [12]:
date_with_most_completed_trips = completed_trip\
    .orderBy("Completed Trips", ascending=False) \
    .select("Date")\
    .first()['Date']

print(f"The date is" ,date_with_most_completed_trips)


The date is 22-Sep-12


#2 What was the highest number of completed trips within a 24-hour period?

In [13]:
highest_number = completed_trip.orderBy('Completed Trips', ascending=False).first()['Completed Trips']
print(highest_number)

248


#3 Which hour of the day had the most requests during the two-week period?



In [14]:
df_most_req = df_updated.groupBy("Time (Local)").sum("Requests ")
df_most_req= df_most_req.withColumnRenamed("sum(Requests )","Requests")
df_most_req=df_most_req.withColumnRenamed("Time (Local)","Time")
df_most_req.orderBy("Requests",ascending=False).first()['Time']

print("The hour with the most request during the two week period is",df_most_req.orderBy("Requests",ascending=False).first()['Time'])

The hour with the most request during the two week period is 23


#4. What percentages of all zeroes during the two-week period occurred on weekends (Friday at 5 pm to Sunday at 3 am)?

In [20]:
from pyspark.sql import functions as F

Finding all zeros

In [71]:
all_zeros = df_updated.agg(F.sum('Zeroes '))
all_zero = (all_zeros.first()['sum(Zeroes )'])

Finding zero entities for weekend

In [88]:
df_with_date = df_updated.withColumn('Date',to_date(col('Date'),"dd-MMM-yy"))
df_weekend = df_with_date.filter(
    (dayofweek(col('Date'))==6 ) & (col('Time (Local)') >=17)|
    (dayofweek(col('Date'))==7) |
    ((dayofweek(col('Date'))==1) & (col('Time (Local)')<3))
)


In [89]:
zero_weekend = df_weekend.agg(F.sum('Zeroes ')).first()['sum(Zeroes )']

In [90]:
Result = (zero_weekend/all_zero) * 100

In [91]:
print("The percentage is", Result)

The percentage is 44.856543037088876
